# CRE office acquisition (Argus-style)

A two-tenant office acquisition modeled lease-by-lease: free rent, anniversary escalations, expense recoveries over stops, TI/LC, probability-weighted rollover, and an exit on forward NOI.

This notebook uses the benchmark model that CFDL validates against an independent reference to the penny (see `benchmarks/`).

In [ ]:
from pathlib import Path
import cfdl_sdk

# Resolve the repo root so the notebook runs from anywhere in a checkout.
ROOT = Path.cwd()
while not (ROOT / "Cargo.toml").exists():
    ROOT = ROOT.parent
PACKS = ROOT / "packs"

## Compile

Compile the model directory to IR.

In [ ]:
model_dir = ROOT / "benchmarks/cre/office_two_tenant"
model = cfdl_sdk.compile(model_dir, packs_dir=PACKS)
print("streams:", len(model.ir["streams"]))

## Run

Run with the benchmark's configuration and apply the `cre` pack's domain metrics.

In [ ]:
results = model.run(
    config=str(model_dir / "run.json"),
    pack="cre",
)
print("status:", results.status, "| warnings:", len(results.warnings))

## Cash flows

The engine returns per-period signed cash flows; `cashflows()` gives a wide DataFrame indexed by period.

In [ ]:
cf = results.cashflows()
print('shape:', cf.shape)
cf.head()

In [ ]:
# Requires the [viz] extra (pip install cfdl-sdk[viz]).
results.plot.cumulative()

## Metrics

Core metrics (NPV/IRR/MOIC/...) plus the pack's domain metrics, with their source labelled.

In [ ]:
results.metrics_frame()

## What-if

Inspect the derived forward-NOI exit value and the DSCR domain metric.

In [ ]:
mf = results.metrics_frame()
mf[mf["metric"].str.contains("dscr|noi|exit", case=False)]